In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)

df_liquidation = pd.read_csv('../data/raw/aave_liquidation_raw.csv')

In [ ]:
aave_pool_address_mapping = {
    '0x7d2768de32b0b80b7a3454c06bdac94a69ddc7a9': 'aave_v2_ethereum',
    '0x87870bca3f3fd6335c3f4ce8392d69350b4fa4e2': 'aave_v3_ethereum'
}

In [ ]:
df_liquidation['block_timestamp'] = pd.to_datetime(df_liquidation["block_timestamp"], errors="coerce", utc=True)
df_liquidation["date"] = df_liquidation["block_timestamp"].dt.date
df_liquidation["minute"] = df_liquidation["block_timestamp"].dt.floor("min")

df_liquidation['record_type'] = 'aave_liquidation'
df_liquidation['protocol_version'] = df_liquidation['address'].map(aave_pool_address_mapping)

df_liquidation['collateral_asset'] = '0x' + df_liquidation['topic1'].str[26:66]
df_liquidation['debt_asset'] = '0x' + df_liquidation['topic2'].str[26:66]
df_liquidation['borrower'] = '0x' + df_liquidation['topic3'].str[26:66]

df_liquidation['debt_to_cover_raw_hex'] = '0x' + df_liquidation['data'].str[2:66]
df_liquidation['liquidated_collateral_amount_raw_hex'] = '0x' + df_liquidation['data'].str[66:130]
df_liquidation['liquidator'] = '0x' + df_liquidation['data'].str[154:194]
df_liquidation['receive_a_token'] = df_liquidation['data'].str[257:258] == '1'

df_liquidation['debt_to_cover_raw'] = df_liquidation['debt_to_cover_raw_hex'].apply(
    lambda h: int(str(h).strip(), 16) if pd.notna(h) else None
)
df_liquidation['liquidated_collateral_amount_raw'] = df_liquidation['liquidated_collateral_amount_raw_hex'].apply(
    lambda h: int(str(h).strip(), 16) if pd.notna(h) else None
)

df_liquidation.head()

In [ ]:
df_liquidation.to_csv('../data/processed/aave_liquidation_processed.csv', index=False)